# Phase 1: Monolithic Bedrock Dry Runs

Run this notebook from top to bottom. It runs the complete AMZN development batch with Amazon Nova Lite, shows every answer, calculates deterministic metrics, and saves the results locally. Gold answers are used only after inference.

## Setup

Load the reusable runtime and evaluation code from `src/`.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

In [2]:
from agent.data import load_question
from agent.monolithic import MonolithicAgentHarness
from agent.observation import format_observable_timeline, format_run_summary
from agent.tools import ProjectTools

/home/atanas-vitanov/Documents/final_project_financial_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from agent.data import load_gold_questions
from evaluation import (
    evaluate_retrieval, evaluate_runs, format_evaluation_summary,
    save_evaluation_results,
)
from judge import evaluate_with_llm_judge

In [4]:
from config import (
    BEDROCK_NOVA_LITE_MODEL_ID, NOVA_LITE_INPUT_USD_PER_MILLION_TOKENS,
    NOVA_LITE_OUTPUT_USD_PER_MILLION_TOKENS,
)

## Development gold answers

These are displayed only for AMZN development analysis. They are never passed to the agent.

In [5]:
ticker = "AMZN"
gold_records = load_gold_questions(ticker)
gold_by_id = {record["question_id"]: record for record in gold_records}

In [6]:
{question_id: record["answer"] for question_id, record in gold_by_id.items()}

{'amzn-01': {'response_type': 'number',
  'rounding': 'exact USD',
  'unit': 'USD',
  'value': '716924000000'},
 'amzn-02': {'response_type': 'number',
  'rounding': 'exact USD',
  'unit': 'USD',
  'value': '79975000000'},
 'amzn-03': {'label': 'AWS',
  'response_type': 'labeled_number',
  'rounding': 'exact USD',
  'unit': 'USD',
  'value': '45606000000'},
 'amzn-04': {'label': 'International',
  'response_type': 'labeled_number',
  'rounding': 'exact USD',
  'unit': 'USD',
  'value': '4750000000'},
 'amzn-05': {'response_type': 'number',
  'rounding': 'nearest 0.01',
  'unit': 'percent',
  'value': '11.16'},
 'amzn-06': {'response_type': 'number',
  'rounding': 'nearest 0.01',
  'unit': 'percent',
  'value': '10.83'},
 'amzn-07': {'response_type': 'number',
  'rounding': 'nearest 0.01',
  'unit': 'percent',
  'value': '19.46'},
 'amzn-08': {'response_type': 'number',
  'rounding': 'nearest 0.01',
  'unit': 'percent',
  'value': '179.62'},
 'amzn-09': {'required_claims': ['operating i

## Preflight

Confirm that Bedrock, Langfuse, and the local Qdrant filing index are available before spending model calls.

In [7]:
harness = MonolithicAgentHarness(
    provider="bedrock", model_id=BEDROCK_NOVA_LITE_MODEL_ID
)
preflight = harness.preflight()

In [8]:
preflight

{'ready': True,
 'provider': 'bedrock',
 'model': 'amazon.nova-lite-v1:0',
 'reasoning_mode': 'none',
 'langfuse': 'authenticated',
 'qdrant_collection': 'sec_filing_chunks_v1'}

## Deterministic tool smoke check

Check fact evidence and Decimal arithmetic before asking the model to use them.

In [9]:
tools = ProjectTools()
facts = tools.lookup_fact(company="AMZN", concept="revenue")

In [10]:
facts

{'status': 'ok',
 'facts': [{'company': 'Amazon.com, Inc.',
   'ticker': 'AMZN',
   'concept': 'revenue',
   'evidence_id': 'fact-amzn-revenue-fy2023',
   'value': '574785000000',
   'unit': 'USD',
   'period': 'FY2023'},
  {'company': 'Amazon.com, Inc.',
   'ticker': 'AMZN',
   'concept': 'revenue',
   'evidence_id': 'fact-amzn-revenue-fy2024',
   'value': '637959000000',
   'unit': 'USD',
   'period': 'FY2024'},
  {'company': 'Amazon.com, Inc.',
   'ticker': 'AMZN',
   'concept': 'revenue',
   'evidence_id': 'fact-amzn-revenue-fy2025',
   'value': '716924000000',
   'unit': 'USD',
   'period': 'FY2025'}]}

In [11]:
calculation = tools.calculate(operation="percentage", inputs=["25", "200"])

In [12]:
calculation

{'status': 'ok', 'result': '12.5'}

## Question 1: direct extraction

Run one question, then inspect its complete audit trail before continuing.

In [13]:
question_1 = load_question("amzn-01")
gold_1 = gold_by_id[question_1.question_id]["answer"]
run_1 = harness.run_question(question_1.question)

In [14]:
run_1.answer

MonolithicAnswer(answer_text="Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.", values=[AnswerValue(label='consolidated net sales', value='716924000000', unit='USD', period='FY2025', source_evidence_ids=['fact-amzn-revenue-fy2025'])], evidence_ids=['fact-amzn-revenue-fy2025'], calculations=[], conclusions=["Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000."])

### Full trace: Question 1

In [15]:
print(format_observable_timeline(run_1.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 1

In [16]:
print(format_run_summary(run_1, gold_1))

Question: What was Amazon.com, Inc.'s consolidated net sales in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.
Gold answer: 716924000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-revenue-fy2025
Latency: 5624.11 ms | Tokens: {'inputTokens': 6385, 'outputTokens': 494, 'totalTokens': 6879} | Retries: 0
Structured-output repairs: 1
Cost: {'input': '0.0003831', 'output': '0.00011856', 'total': '0.00050166'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/6349266c6d78c691c8cc3f192234d71e


### Cost breakdown: Question 1

Nova Lite on-demand text pricing in `us-east-1`, frozen on 2026-08-24.

In [17]:
input_cost = run_1.token_usage["inputTokens"] * NOVA_LITE_INPUT_USD_PER_MILLION_TOKENS / 1_000_000
output_cost = run_1.token_usage["outputTokens"] * NOVA_LITE_OUTPUT_USD_PER_MILLION_TOKENS / 1_000_000
input_cost + output_cost

Decimal('0.00050166')

## Question 2: single formula

Run one formula question, then inspect it before moving on.

In [18]:
question_2 = load_question("amzn-05")
gold_2 = gold_by_id[question_2.question_id]["answer"]
run_2 = harness.run_question(question_2.question)

### Full trace: Question 2

In [19]:
print(format_observable_timeline(run_2.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 2

In [20]:
print(format_run_summary(run_2, gold_2))

Question: What was Amazon.com, Inc.'s operating margin in fiscal 2025?
Model answer: Amazon.com, Inc.'s operating margin in fiscal 2025 was 11.16 percent.
Gold answer: 11.16
Gold values: -
Tools: lookup_fact (ok), lookup_fact (ok), calculate (ok)
Evidence: fact-amzn-operating-income-fy2025, fact-amzn-revenue-fy2025
Latency: 4791.55 ms | Tokens: {'inputTokens': 6799, 'outputTokens': 412, 'totalTokens': 7211} | Retries: 0
Structured-output repairs: 0
Cost: {'input': '0.00040794', 'output': '0.00009888', 'total': '0.00050682'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/6a937c6e5d5e0e5ead0cceeb9bd0c9f1


## Question 3: period comparison

This example combines filing search, normalized facts, and calculations.

In [21]:
question_3 = load_question("amzn-12")
gold_3 = gold_by_id[question_3.question_id]["answer"]
run_3 = harness.run_question(question_3.question)

### Full trace: Question 3

In [22]:
print(format_observable_timeline(run_3.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 3

In [23]:
print(format_run_summary(run_3, gold_3))

Question: From fiscal 2024 to fiscal 2025, how did AWS operating income growth compare with Amazon's consolidated operating income growth?
Model answer: From fiscal 2024 to fiscal 2025, AWS operating income grew by 118.63%, while Amazon's consolidated operating income grew by 116.59%. AWS operating income grew at a higher rate than Amazon's consolidated operating income.
Gold answer: consolidated operating income grew faster than AWS operating income; the growth-rate difference was 2.10 percentage points
Gold values: AWS operating income growth: 14.49 percent, consolidated operating income growth: 16.59 percent
Tools: search_filing (ok), search_filing (ok), calculate (ok), calculate (ok)
Evidence: filing-amzn-2025-page-0068-table-01
Latency: 14792.21 ms | Tokens: {'inputTokens': 27628, 'outputTokens': 664, 'totalTokens': 28292} | Retries: 0
Structured-output repairs: 0
Cost: {'input': '0.00165768', 'output': '0.00015936', 'total': '0.00181704'}
Trace: https://cloud.langfuse.com/project

## Question 4: multi-step calculation

The final dry run checks a longer chain before the complete batch.

In [24]:
question_4 = load_question("amzn-16")
gold_4 = gold_by_id[question_4.question_id]["answer"]
run_4 = harness.run_question(question_4.question)

### Full trace: Question 4

In [25]:
print(format_observable_timeline(run_4.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 4

In [26]:
print(format_run_summary(run_4, gold_4))

Question: Did Amazon's net sales, operating income, operating cash flow, and total assets grow at similar rates from fiscal 2024 to fiscal 2025?
Model answer: Amazon's net sales, operating income, operating cash flow, and total assets did not grow at similar rates from fiscal 2024 to fiscal 2025.
Gold answer: all four measures increased; total assets grew fastest; net sales grew slowest; the spread between the fastest and slowest growth rates was 18.53 percentage points
Gold values: net sales growth: 12.38 percent, operating income growth: 16.59 percent, operating cash flow growth: 20.40 percent, total assets growth: 30.91 percent
Tools: lookup_fact (ok), lookup_fact (ok), lookup_fact (ok), lookup_fact (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok)
Evidence: fact-amzn-revenue-fy2024, fact-amzn-revenue-fy2025, fact-amzn-operating-income-fy2024, fact-amzn-operating-income-fy2025, fact-amzn-operating-cash-flow-fy2024, fact-amzn-operati

## Full AMZN development batch

Run all 16 development questions with Nova Lite. Each attempt is appended to the official local JSONL run log.

In [27]:
batch_runs = harness.run_batch(ticker)
assert [run.question.question_id for run in batch_runs] == [gold["question_id"] for gold in gold_records]

In [28]:
[(run.question.question_id, run.status) for run in batch_runs]

[('amzn-01', 'success'),
 ('amzn-02', 'success'),
 ('amzn-03', 'success'),
 ('amzn-04', 'success'),
 ('amzn-05', 'success'),
 ('amzn-06', 'success'),
 ('amzn-07', 'success'),
 ('amzn-08', 'success'),
 ('amzn-09', 'success'),
 ('amzn-10', 'success'),
 ('amzn-11', 'success'),
 ('amzn-12', 'success'),
 ('amzn-13', 'success'),
 ('amzn-14', 'success'),
 ('amzn-15', 'success'),
 ('amzn-16', 'success')]

## Inspect every answer

The compact view makes incorrect values, missing evidence, failed calls, and unexpectedly expensive questions visible before looking at aggregate scores.

In [31]:
for run in batch_runs:
    gold = gold_by_id[run.question.question_id]["answer"]
    print(format_run_summary(run, gold))
    print("-" * 88)

Question: What was Amazon.com, Inc.'s consolidated net sales in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.
Gold answer: 716924000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-revenue-fy2025
Latency: 2782.03 ms | Tokens: {'inputTokens': 3977, 'outputTokens': 252, 'totalTokens': 4229} | Retries: 0
Structured-output repairs: 0
Cost: {'input': '0.00023862', 'output': '0.00006048', 'total': '0.00029910'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/4775e63fda5f1c5f7813f7be496c6a88
----------------------------------------------------------------------------------------
Question: What was Amazon.com, Inc.'s consolidated operating income in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated operating income in fiscal 2025 was $79,975,000,000.
Gold answer: 79975000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-operating-income-fy2025
Latency: 2276.84 ms | Tokens:

## Deterministic evaluation

Gold values, evidence IDs, calculation steps, required tools, and saved tool events are compared with ordinary Python. Values must exactly match gold after scale normalization; a difference of 0.001 fails. A conflicting unit or an unexplained number also fails.

Retrieval is measured from saved tool events. Haiku 4.5 optionally adds qualitative 1–5 workflow scores without overriding deterministic numerical correctness.

In [32]:
question_metrics, summary = evaluate_runs(batch_runs, gold_records)
retrieval_metrics, retrieval_summary = evaluate_retrieval(batch_runs, gold_records)
summary.update(retrieval_summary)

## Haiku 4.5 workflow judge

The judge is reused for the batch, uses Haiku 4.5 extended thinking, and cannot override an exact numeric failure.

In [33]:
judge_metrics, judge_summary = evaluate_with_llm_judge(
    batch_runs, gold_records, question_metrics
)
summary.update(judge_summary)

In [34]:
judge_metrics

[{'run_id': 'amzn-01-20260830T021902723186Z',
  'question_id': 'amzn-01',
  'category': 'direct_extraction',
  'judge_model': 'us.anthropic.claude-haiku-4-5-20251001-v1:0',
  'llm_answer_correctness_grade': 5,
  'workflow_correctness': 5,
  'chronological_coherence': 5,
  'workflow_efficiency': 5,
  'auditability': 5,
  'explanation': 'The answer is numerically correct (716,924,000,000 USD), matches the gold requirement exactly, and uses the correct period (FY2025) and concept (consolidated net sales). The workflow is optimally efficient, using a single lookup_fact call that directly retrieved the required evidence (fact-amzn-revenue-fy2025). The trace is fully auditable with clear tool parameters and result documentation.',
  'failure': None},
 {'run_id': 'amzn-02-20260830T021905506191Z',
  'question_id': 'amzn-02',
  'category': 'direct_extraction',
  'judge_model': 'us.anthropic.claude-haiku-4-5-20251001-v1:0',
  'llm_answer_correctness_grade': 5,
  'workflow_correctness': 5,
  'chr

### Questions needing review

Only runs with a failed deterministic check appear here.

In [35]:
CHECKS = ("numerical_correct", "evidence_valid", "evidence_complete",
          "value_evidence_valid", "periods_valid", "audit_trail_complete")
failed_question_metrics = [row for row in question_metrics if row["status"] != "success"
                           or any(row[name] is not True for name in CHECKS)]
failed_question_metrics

[{'run_id': 'amzn-03-20260830T021907783925Z',
  'question_id': 'amzn-03',
  'category': 'direct_extraction',
  'status': 'success',
  'answer_text': 'The AWS segment generated the most operating income in fiscal 2025, with an amount of $45,606 million.',
  'numerical_correct': False,
  'numerical_issues': ['AWS=45606000000 USD'],
  'unexpected_values': ['Operating income of AWS segment=45606 USD'],
  'evidence_valid': True,
  'invalid_evidence_ids': [],
  'evidence_complete': True,
  'missing_evidence_ids': [],
  'value_evidence_valid': False,
  'invalid_value_evidence': ['Operating income of AWS segment'],
  'periods_valid': True,
  'formula_correct': None,
  'formatting_correct': True,
  'unsupported_values': ['Operating income of AWS segment: 45606 USD'],
  'audit_trail_complete': False,
  'required_tools': ['search_filing'],
  'invoked_tools': ['calculate', 'search_filing'],
  'tool_precision': 0.5,
  'tool_recall': 1.0,
  'tool_f1': 0.6666666666666666,
  'tool_exact_match': False,

### Metrics by question type

Category scores show whether errors are concentrated in extraction, one formula, comparisons, or longer calculation chains.

In [36]:
summary["by_category"]

{'direct_extraction': {'run_count': 4,
  'numerical_correctness_rate': 0.5,
  'evidence_completeness_rate': 1.0,
  'formula_correctness_rate': None,
  'formatting_correctness_rate': 1.0},
 'multi_step': {'run_count': 4,
  'numerical_correctness_rate': 0.0,
  'evidence_completeness_rate': 0.75,
  'formula_correctness_rate': 0.0,
  'formatting_correctness_rate': 1.0},
 'period_comparison': {'run_count': 4,
  'numerical_correctness_rate': 0.0,
  'evidence_completeness_rate': 0.75,
  'formula_correctness_rate': 0.0,
  'formatting_correctness_rate': 1.0},
 'single_formula': {'run_count': 4,
  'numerical_correctness_rate': 1.0,
  'evidence_completeness_rate': 1.0,
  'formula_correctness_rate': 1.0,
  'formatting_correctness_rate': 1.0}}

### Final metrics

Higher is better except for unsupported-number rate and irrelevant retrieval rate. Evidence validity checks whether cited IDs were actually returned during the run; evidence completeness checks whether all frozen gold evidence was cited.

In [37]:
print(format_evaluation_summary(summary))

Runs: 16
run_success_rate: 100.00%
numerical_correctness_rate: 37.50%
evidence_validity_rate: 100.00%
evidence_completeness_rate: 87.50%
formula_correctness_rate: 33.33%
formatting_correctness_rate: 100.00%
unsupported_number_rate: 24.00%
audit_trail_completeness_rate: 37.50%
tool_precision: 96.55%
tool_recall: 93.33%
tool_f1: 94.92%
tool_exact_match_rate: 81.25%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 100.00%
conditional_execution_rate: 93.75%
evidence_recall_at_5: 50.00%
irrelevant_retrieval_rate_at_5: 90.00%
judge_success_rate: 100.00%
structured_output_repair_rate: 12.50%
Average latency: 5821.90 ms
Input tokens: 176499
Output tokens: 10908
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 2
Infrastructure failures: 0
Estimated cost: $0.01320786
Judge llm answer correctness grade: 3.06/5
Judge workflow correctness: 3.06/5
Judge chronological coherence: 4.75/5
Judge workflow efficiency: 4.12/5
Judge auditability: 4.50/5


## Save this batch

The folder name contains the run date, model, and company ticker. It contains complete JSONL run records plus CSV and JSON metric files.

In [38]:
results_dir = save_evaluation_results(
    batch_runs, question_metrics, summary, retrieval_metrics,
    judge_rows=judge_metrics or None,
)
results_dir

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-08-30T022415874493Z-monolithic-amazon-nova-lite-v1-0-AMZN')

In [39]:
sorted(path.name for path in results_dir.iterdir())

['judge_records.jsonl',
 'question_metrics.csv',
 'retrieval_metrics.csv',
 'runs.jsonl',
 'summary.json']

## Failures

Provider and tool failures remain experimental results. They are displayed rather than silently rerun or repaired.

In [40]:
[
    (run.question.question_id, run.failure_category, run.failure_message)
    for run in batch_runs if run.status == "failure"
]

[]

# Validation batch: Alphabet (GOOGL)

Run the frozen validation company after reviewing the AMZN development results.

In [41]:
validation_ticker = "GOOGL"
validation_gold_records = load_gold_questions(validation_ticker)

In [42]:
validation_runs = harness.run_batch(validation_ticker)
assert len(validation_runs) == len(validation_gold_records)

exception=<EventStreamError> | event loop cycle failed


In [43]:
validation_question_metrics, validation_summary = evaluate_runs(
    validation_runs, validation_gold_records
)
validation_retrieval_metrics, validation_retrieval_summary = evaluate_retrieval(
    validation_runs, validation_gold_records
)
validation_summary.update(validation_retrieval_summary)
validation_judge_metrics, validation_judge_summary = evaluate_with_llm_judge(
    validation_runs, validation_gold_records, validation_question_metrics
)
validation_summary.update(validation_judge_summary)

In [44]:
print(format_evaluation_summary(validation_summary))

Runs: 16
run_success_rate: 93.75%
numerical_correctness_rate: 43.75%
evidence_validity_rate: 87.50%
evidence_completeness_rate: 62.50%
formula_correctness_rate: 33.33%
formatting_correctness_rate: 93.75%
unsupported_number_rate: 25.00%
audit_trail_completeness_rate: 25.00%
tool_precision: 82.76%
tool_recall: 80.00%
tool_f1: 81.36%
tool_exact_match_rate: 56.25%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 97.14%
conditional_execution_rate: 77.08%
evidence_recall_at_5: 100.00%
irrelevant_retrieval_rate_at_5: 70.00%
judge_success_rate: 100.00%
structured_output_repair_rate: 12.50%
Average latency: 7900.98 ms
Input tokens: 207812
Output tokens: 11890
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 2
Infrastructure failures: 0
Estimated cost: $0.01532232
Judge llm answer correctness grade: 3.00/5
Judge workflow correctness: 2.94/5
Judge chronological coherence: 4.44/5
Judge workflow efficiency: 3.75/5
Judge auditability: 3.81/5


In [45]:
validation_results_dir = save_evaluation_results(
    validation_runs, validation_question_metrics, validation_summary,
    validation_retrieval_metrics, judge_rows=validation_judge_metrics,
)
validation_results_dir

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-08-30T022955351242Z-monolithic-amazon-nova-lite-v1-0-GOOGL')